<a href="https://colab.research.google.com/github/lightcode01-oss/IYB_INTERNSHIP/blob/main/Day_12_SVM_Classification_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Run from a cloned repository, locally or in Google Colab.
from pathlib import Path
import os, sys, subprocess
root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if not (root / "src").exists():
    target = root / "IYB_INTERNSHIP"
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/lightcode01-oss/IYB_INTERNSHIP.git", str(target)], check=True)
    root = target
os.chdir(root)
sys.path.insert(0, str(root))
print("Repository ready:", root.name)


Repository ready: IYB_INTERNSHIP


# Day 12 - Support Vector Machine (SVM) Classification Model

## Project
AI-Based Student Performance Prediction System

## Internship
YBI Foundation - Python AI & Data Science Internship

## Objective
The objective of Day 12 is to train and evaluate a Support Vector Machine
(SVM) classification model for predicting student performance categories.

The SVM model will be trained as an untuned baseline model and compared
with the classification models developed during previous days:

- Logistic Regression
- Decision Tree
- Random Forest
- K-Nearest Neighbors (KNN)
- Support Vector Machine (SVM)

The evaluation will focus especially on Macro F1-score because the project
contains multiple performance categories and each class should be considered
during evaluation.

## Day 12 Learning Goals

By the end of this notebook, the following tasks will be completed:

1. Load the student performance dataset.
2. Recreate the project-defined performance categories.
3. Separate features and target.
4. Prevent target leakage by removing `G3` from model inputs.
5. Split the dataset using a stratified train-test split.
6. Prepare categorical and numerical features.
7. Apply scaling to numerical and ordinal features.
8. Build an SVM classification pipeline.
9. Train the baseline SVM model.
10. Evaluate the model using multiple classification metrics.
11. Analyze the confusion matrix.
12. Compare SVM with the previous four models.
13. Rank all five models using Macro F1-score.
14. Save the model comparison results.

## Why Support Vector Machine?

Support Vector Machine (SVM) is a supervised machine learning algorithm
that can be used for classification problems.

SVM attempts to find a decision boundary that separates different classes
while maximizing the margin between them.

For this project, SVM is useful because:

- The problem is a multiclass classification problem.
- The dataset contains a mixture of numerical and categorical features.
- Feature scaling can be applied before SVM training.
- SVM can model non-linear class boundaries when using an RBF kernel.

The SVM used today is a baseline model. Hyperparameter tuning will be
performed later in the project.

In [2]:
# Install the dataset library
# Dependencies: install requirements-dev.txt once before running.


In [3]:
# Import required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("All libraries imported successfully.")

All libraries imported successfully.


In [4]:
# Display library versions

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

import sklearn
print("Scikit-learn version:", sklearn.__version__)

NumPy version: 2.5.4
Pandas version: 3.1.0
Scikit-learn version: 1.9.0


## 1. Load Dataset

The UCI Student Performance dataset is used for this project.

The dataset contains demographic, social, family, study-related and
academic information about students.

The final grade `G3` is used to create the project-defined performance
category.

In [5]:
# Fetch the UCI Student Performance dataset

student_performance = fetch_ucirepo(id=320)

X_original = student_performance.data.features
y_original = student_performance.data.targets

# Create working dataframe
df = X_original.copy()

# Add final grade
df["G3"] = y_original["G3"]

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully.
Dataset shape: (649, 31)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,no,no,4,3,4,1,1,3,4,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,yes,no,5,3,3,1,1,3,2,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,yes,no,4,3,2,2,3,3,6,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,yes,yes,3,2,2,1,1,5,0,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,no,no,4,3,2,1,2,5,0,13


In [6]:
# Function to create the project-defined performance category

def performance_category(grade):
    if grade < 10:
        return "Low"
    elif grade < 15:
        return "Medium"
    else:
        return "High"


# Create target column
df["performance_category"] = df["G3"].apply(performance_category)

print("Performance category created successfully.")

display(
    df[["G3", "performance_category"]].head(10)
)

Performance category created successfully.


,G3,performance_category
0,11,Medium
1,11,Medium
2,12,Medium
3,14,Medium
4,13,Medium
5,13,Medium
6,13,Medium
7,13,Medium
8,17,High
9,13,Medium


In [7]:
# Check target distribution

print("Performance category distribution:")

display(
    df["performance_category"]
    .value_counts()
    .reindex(["Low", "Medium", "High"])
)

Performance category distribution:


performance_category
Low       100
Medium    418
High      131
Name: count, dtype: int64

In [8]:
# Visualize target distribution

plt.figure(figsize=(7, 5))

sns.countplot(
    data=df,
    x="performance_category",
    order=["Low", "Medium", "High"]
)

plt.title("Student Performance Category Distribution")
plt.xlabel("Performance Category")
plt.ylabel("Number of Students")

plt.show()

C:\Users\ABHINAV DASH\AppData\Local\Temp\ipykernel_21592\3781458659.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Separate Features and Target

The target variable is:

`performance_category`

The column `G3` must not be used as an input feature.

This is important because the performance category is directly created
from `G3`.

If `G3` were included during model training, the model would receive
information that directly determines the target. This would create
target leakage and produce an unreliable evaluation.

Therefore, `G3` is removed before model training.

In [9]:
# Separate target from input data

target = "performance_category"

X = df.drop(columns=[target])
y = df[target]

# Remove G3 to prevent target leakage
X_model = X.drop(columns=["G3"])

print("Original feature count:", X.shape[1])
print("Model feature count:", X_model.shape[1])

print("\nTarget column:", target)
print("G3 present in model features:", "G3" in X_model.columns)

print("\nTarget distribution:")
display(y.value_counts().reindex(["Low", "Medium", "High"]))

Original feature count: 31
Model feature count: 30

Target column: performance_category
G3 present in model features: False

Target distribution:


performance_category
Low       100
Medium    418
High      131
Name: count, dtype: int64

In [10]:
# Display model input columns

print("Features used for model training:\n")

for i, column in enumerate(X_model.columns, start=1):
    print(f"{i}. {column}")

Features used for model training:

1. school
2. sex
3. age
4. address
5. famsize
6. Pstatus
7. Medu
8. Fedu
9. Mjob
10. Fjob
11. reason
12. guardian
13. traveltime
14. studytime
15. failures
16. schoolsup
17. famsup
18. paid
19. activities
20. nursery
21. higher
22. internet
23. romantic
24. famrel
25. freetime
26. goout
27. Dalc
28. Walc
29. health
30. absences


## 4. Define Feature Groups

The features are divided into different groups based on their data type
and meaning.

### Numerical Features
- age
- absences

### Ordinal Features
These contain ordered values:

- Medu
- Fedu
- traveltime
- studytime
- failures
- famrel
- freetime
- goout
- Dalc
- Walc
- health

### Binary Features
- school
- sex
- address
- famsize
- Pstatus
- schoolsup
- famsup
- paid
- activities
- nursery
- higher
- internet
- romantic

### Nominal Features
- Mjob
- Fjob
- reason
- guardian

Categorical variables will be converted using One-Hot Encoding.

Numerical and ordinal features will be standardized because SVM is
sensitive to feature scale.

In [11]:
# Define feature groups

numerical_features = [
    "age",
    "absences"
]

ordinal_features = [
    "Medu",
    "Fedu",
    "traveltime",
    "studytime",
    "failures",
    "famrel",
    "freetime",
    "goout",
    "Dalc",
    "Walc",
    "health"
]

binary_features = [
    "school",
    "sex",
    "address",
    "famsize",
    "Pstatus",
    "schoolsup",
    "famsup",
    "paid",
    "activities",
    "nursery",
    "higher",
    "internet",
    "romantic"
]

nominal_features = [
    "Mjob",
    "Fjob",
    "reason",
    "guardian"
]

categorical_features = binary_features + nominal_features

print("Numerical features:", len(numerical_features))
print("Ordinal features:", len(ordinal_features))
print("Binary features:", len(binary_features))
print("Nominal features:", len(nominal_features))
print("Total categorical features:", len(categorical_features))

Numerical features: 2
Ordinal features: 11
Binary features: 13
Nominal features: 4
Total categorical features: 17


In [12]:
# Verify that all feature groups are present

all_grouped_features = (
    numerical_features
    + ordinal_features
    + categorical_features
)

missing_features = [
    feature for feature in all_grouped_features
    if feature not in X_model.columns
]

print("Missing features:", missing_features)

if len(missing_features) == 0:
    print("All feature groups are valid.")
else:
    print("Please check the feature definitions.")

Missing features: []
All feature groups are valid.


## 5. Train-Test Split

The dataset will be divided into:

- 80% training data
- 20% testing data

`stratify=y` is used so that the class distribution remains approximately
consistent between the training and testing datasets.

The same random state used in previous model notebooks is retained:

`random_state = 42`

This makes model comparisons more consistent.

In [13]:
# Create stratified train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining target distribution:")
display(
    y_train.value_counts(normalize=True)
    .reindex(["Low", "Medium", "High"])
)

print("\nTesting target distribution:")
display(
    y_test.value_counts(normalize=True)
    .reindex(["Low", "Medium", "High"])
)

Training samples: 519
Testing samples: 130

Training target distribution:


performance_category
Low       0.154143
Medium    0.643545
High      0.202312
Name: proportion, dtype: float64


Testing target distribution:


performance_category
Low       0.153846
Medium    0.646154
High      0.200000
Name: proportion, dtype: float64

## 6. Build SVM Preprocessing Pipeline

SVM uses distances and margins to identify class boundaries.

Therefore, numerical and ordinal features should be placed on a comparable
scale.

The preprocessing pipeline will perform:

### Categorical Features
One-Hot Encoding

### Numerical + Ordinal Features
Standard Scaling

The preprocessing will be fitted only on the training data through the
Scikit-learn Pipeline.

This prevents information from the test dataset from influencing
preprocessing.

In [14]:
# Create preprocessing components

categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

numerical_scaler = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_encoder,
            categorical_features
        ),
        (
            "numerical",
            numerical_scaler,
            numerical_features + ordinal_features
        )
    ],
    remainder="drop"
)

print("SVM preprocessing configuration created successfully.")

SVM preprocessing configuration created successfully.


## 7. Create Baseline SVM Model

A Support Vector Classifier with an RBF kernel will be used.

The model is intentionally kept as a baseline.

No hyperparameter tuning is performed today.

The parameters such as `C` and `gamma` will be investigated during the
future hyperparameter tuning stage.

In [15]:
# Create baseline SVM classifier

svm_model = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

print("Baseline SVM model created.")
print(svm_model)

Baseline SVM model created.
SVC(probability=True, random_state=42)


In [16]:
# Create complete SVM pipeline

svm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", svm_model)
])

print("SVM pipeline created successfully.")

SVM pipeline created successfully.


In [17]:
# Train the SVM pipeline

svm_pipeline.fit(X_train, y_train)

print("SVM model trained successfully.")

SVM model trained successfully.


C:\Users\ABHINAV DASH\Documents\Codex\2026-11-02\and\work\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [18]:
# Generate predictions

y_train_pred = svm_pipeline.predict(X_train)
y_test_pred = svm_pipeline.predict(X_test)

print("Training predictions generated:", len(y_train_pred))
print("Testing predictions generated:", len(y_test_pred))

Training predictions generated: 519
Testing predictions generated: 130


## 8. Evaluate SVM Model

The following metrics will be calculated:

- Accuracy
- Macro Precision
- Macro Recall
- Macro F1-score

Macro averaging gives equal importance to each performance category.

This is useful because the project is a multiclass classification problem.

In [19]:
# Calculate SVM evaluation metrics

svm_accuracy = accuracy_score(y_test, y_test_pred)

svm_precision = precision_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

svm_recall = recall_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

svm_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

svm_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1"
    ],
    "Score": [
        svm_accuracy,
        svm_precision,
        svm_recall,
        svm_f1
    ]
})

display(svm_results)

,Metric,Score
0,Accuracy,0.661538
1,Macro Precision,0.469577
2,Macro Recall,0.379365
3,Macro F1,0.346825


In [20]:
# Display formatted metrics

print("SVM Test Performance")
print("-" * 35)

print(f"Accuracy        : {svm_accuracy:.4f}")
print(f"Macro Precision : {svm_precision:.4f}")
print(f"Macro Recall    : {svm_recall:.4f}")
print(f"Macro F1        : {svm_f1:.4f}")

SVM Test Performance
-----------------------------------
Accuracy        : 0.6615
Macro Precision : 0.4696
Macro Recall    : 0.3794
Macro F1        : 0.3468


## 9. Classification Report

The classification report provides class-level precision, recall and F1-score.

This helps identify whether the SVM performs similarly across Low, Medium
and High performance categories.

In [21]:
# Classification report

print("SVM Classification Report")
print("=" * 60)

print(
    classification_report(
        y_test,
        y_test_pred,
        labels=["Low", "Medium", "High"],
        zero_division=0
    )
)

SVM Classification Report
              precision    recall  f1-score   support

         Low       0.75      0.15      0.25        20
      Medium       0.66      0.99      0.79        84
        High       0.00      0.00      0.00        26

    accuracy                           0.66       130
   macro avg       0.47      0.38      0.35       130
weighted avg       0.54      0.66      0.55       130



## 10. Confusion Matrix

The confusion matrix shows how many students were correctly or incorrectly
classified into each performance category.

Rows represent the actual category.

Columns represent the predicted category.

In [22]:
# Calculate confusion matrix

cm_svm = confusion_matrix(
    y_test,
    y_test_pred,
    labels=["Low", "Medium", "High"]
)

cm_svm_df = pd.DataFrame(
    cm_svm,
    index=["Actual Low", "Actual Medium", "Actual High"],
    columns=["Predicted Low", "Predicted Medium", "Predicted High"]
)

display(cm_svm_df)

,Predicted Low,Predicted Medium,Predicted High
Actual Low,3,17,0
Actual Medium,1,83,0
Actual High,0,26,0


In [23]:
# Visualize confusion matrix

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm_svm_df,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("SVM Confusion Matrix")
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")

plt.show()

C:\Users\ABHINAV DASH\AppData\Local\Temp\ipykernel_21592\4203813659.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Training vs Testing Accuracy

Training accuracy and testing accuracy are compared to get an initial
indication of how the baseline SVM behaves on seen and unseen data.

A very large difference between the two values may indicate possible
overfitting.

This comparison is only an initial diagnostic and is not a substitute for
cross-validation.

In [24]:
# Calculate training and testing accuracy

svm_train_accuracy = accuracy_score(
    y_train,
    y_train_pred
)

svm_test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

accuracy_comparison = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "Accuracy": [
        svm_train_accuracy,
        svm_test_accuracy
    ]
})

display(accuracy_comparison)

,Dataset,Accuracy
0,Training,0.730250
1,Testing,0.661538


In [25]:
# Plot training vs testing accuracy

plt.figure(figsize=(7, 5))

sns.barplot(
    data=accuracy_comparison,
    x="Dataset",
    y="Accuracy"
)

plt.ylim(0, 1)
plt.title("SVM Training vs Testing Accuracy")
plt.ylabel("Accuracy")

plt.show()

C:\Users\ABHINAV DASH\AppData\Local\Temp\ipykernel_21592\1567655673.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Prediction Probability

The SVM model was configured with `probability=True`.

This allows the model to provide estimated class probabilities for
individual predictions.

These values can later be useful when building a student performance
prediction interface.

In [26]:
# Generate probability predictions

svm_probabilities = svm_pipeline.predict_proba(X_test)

probability_df = pd.DataFrame(
    svm_probabilities,
    columns=svm_pipeline.classes_
)

probability_df["Actual"] = y_test.reset_index(drop=True)
probability_df["Predicted"] = y_test_pred

probability_df["Confidence"] = probability_df[
    svm_pipeline.classes_
].max(axis=1)

display(probability_df.head(10))

,High,Low,Medium,Actual,Predicted,Confidence
0,0.084790,0.175560,0.739650,Medium,Medium,0.739650
1,0.096435,0.220613,0.682951,Low,Medium,0.682951
2,0.254080,0.143659,0.602261,Medium,Medium,0.602261
3,0.098742,0.245959,0.655299,Low,Medium,0.655299
4,0.177270,0.435098,0.387632,Medium,Medium,0.435098
5,0.133852,0.123962,0.742185,Medium,Medium,0.742185
6,0.182912,0.114845,0.702242,High,Medium,0.702242
7,0.289060,0.061246,0.649695,Medium,Medium,0.649695
8,0.118462,0.129512,0.752027,Medium,Medium,0.752027
9,0.062812,0.576010,0.361178,Low,Low,0.576010


In [27]:
# Display average prediction confidence

average_confidence = probability_df["Confidence"].mean()

print(
    f"Average SVM prediction confidence: "
    f"{average_confidence:.4f}"
)

Average SVM prediction confidence: 0.6743


In [28]:
# Plot confidence distribution

plt.figure(figsize=(8, 5))

sns.histplot(
    probability_df["Confidence"],
    bins=10,
    kde=True
)

plt.title("SVM Prediction Confidence Distribution")
plt.xlabel("Prediction Confidence")
plt.ylabel("Number of Predictions")

plt.show()

C:\Users\ABHINAV DASH\AppData\Local\Temp\ipykernel_21592\1699430283.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Predicted Class Distribution

The predicted class distribution is compared with the actual test-set
distribution.

This provides an additional view of how the model distributes predictions
across the three performance categories.

In [29]:
# Compare actual and predicted class counts

actual_counts = (
    y_test
    .value_counts()
    .reindex(["Low", "Medium", "High"], fill_value=0)
)

predicted_counts = (
    pd.Series(y_test_pred)
    .value_counts()
    .reindex(["Low", "Medium", "High"], fill_value=0)
)

distribution_comparison = pd.DataFrame({
    "Actual": actual_counts,
    "Predicted": predicted_counts
})

display(distribution_comparison)

,Actual,Predicted
Low,20,4
Medium,84,126
High,26,0


In [30]:
# Plot actual vs predicted class distribution

distribution_comparison.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Actual vs Predicted Performance Categories")
plt.xlabel("Performance Category")
plt.ylabel("Number of Students")

plt.xticks(rotation=0)
plt.show()

C:\Users\ABHINAV DASH\AppData\Local\Temp\ipykernel_21592\288753931.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# 14. Compare SVM with Previous Models

The project has now developed five baseline classification models:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. K-Nearest Neighbors
5. Support Vector Machine

For a fair comparison, the same:

- Dataset
- Target definition
- Target leakage prevention
- Train-test split
- Evaluation metrics

are used.

The previous models are recreated in this notebook so that the complete
comparison can be reproduced from one standalone notebook.

In [31]:
# Create the previous baseline models

logistic_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

decision_tree_model = DecisionTreeClassifier(
    random_state=42
)

random_forest_model = RandomForestClassifier(
    random_state=42
)

knn_model = KNeighborsClassifier(
    n_neighbors=5
)

print("Previous baseline models created.")

Previous baseline models created.


In [32]:
# Create preprocessing for models that do not require scaling

categorical_encoder_basic = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

preprocessor_basic = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_encoder_basic,
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features + ordinal_features
        )
    ],
    remainder="drop"
)

print("Basic preprocessing pipeline created.")

Basic preprocessing pipeline created.


In [33]:
# Create model pipelines

logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor_basic),
    ("classifier", logistic_model)
])

decision_tree_pipeline = Pipeline([
    ("preprocessor", preprocessor_basic),
    ("classifier", decision_tree_model)
])

random_forest_pipeline = Pipeline([
    ("preprocessor", preprocessor_basic),
    ("classifier", random_forest_model)
])

# KNN requires scaled numerical and ordinal features

knn_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features + ordinal_features
        )
    ],
    remainder="drop"
)

knn_pipeline = Pipeline([
    ("preprocessor", knn_preprocessor),
    ("classifier", knn_model)
])

print("All model pipelines created.")

All model pipelines created.


In [34]:
# Train previous models

logistic_pipeline.fit(X_train, y_train)
decision_tree_pipeline.fit(X_train, y_train)
random_forest_pipeline.fit(X_train, y_train)
knn_pipeline.fit(X_train, y_train)

print("Previous baseline models trained successfully.")

Previous baseline models trained successfully.


In [35]:
# Generate predictions from previous models

logistic_pred = logistic_pipeline.predict(X_test)
decision_tree_pred = decision_tree_pipeline.predict(X_test)
random_forest_pred = random_forest_pipeline.predict(X_test)
knn_pred = knn_pipeline.predict(X_test)

print("Predictions generated for all previous models.")

Predictions generated for all previous models.


In [36]:
# Function to calculate model metrics

def calculate_model_metrics(model_name, y_true, y_pred):

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    }

In [37]:
# Build comparison table

comparison_results = [
    calculate_model_metrics(
        "Logistic Regression",
        y_test,
        logistic_pred
    ),

    calculate_model_metrics(
        "Decision Tree",
        y_test,
        decision_tree_pred
    ),

    calculate_model_metrics(
        "Random Forest",
        y_test,
        random_forest_pred
    ),

    calculate_model_metrics(
        "KNN",
        y_test,
        knn_pred
    ),

    calculate_model_metrics(
        "SVM",
        y_test,
        y_test_pred
    )
]

model_comparison = pd.DataFrame(comparison_results)

display(model_comparison)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Logistic Regression,0.669231,0.599889,0.507631,0.510613
1,Decision Tree,0.530769,0.446826,0.463919,0.450885
2,Random Forest,0.623077,0.437098,0.355678,0.327929
3,KNN,0.638462,0.552284,0.472527,0.493093
4,SVM,0.661538,0.469577,0.379365,0.346825


## 15. Rank Baseline Models

The models will be ranked according to Macro F1-score.

Macro F1 is selected as the primary comparison metric because it calculates
F1-score independently for each class and then gives equal importance to
the classes.

This helps avoid relying only on overall accuracy.

In [38]:
# Rank models by Macro F1

model_ranking = model_comparison.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

model_ranking.index = model_ranking.index + 1

display(model_ranking)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
1,Logistic Regression,0.669231,0.599889,0.507631,0.510613
2,KNN,0.638462,0.552284,0.472527,0.493093
3,Decision Tree,0.530769,0.446826,0.463919,0.450885
4,SVM,0.661538,0.469577,0.379365,0.346825
5,Random Forest,0.623077,0.437098,0.355678,0.327929


In [39]:
# Identify the best baseline model by Macro F1

best_model = model_ranking.iloc[0]["Model"]
best_macro_f1 = model_ranking.iloc[0]["Macro F1"]

print("Best baseline model by Macro F1:")
print(best_model)
print(f"Macro F1: {best_macro_f1:.4f}")

Best baseline model by Macro F1:
Logistic Regression
Macro F1: 0.5106


In [40]:
# Plot Macro F1 comparison

plt.figure(figsize=(10, 6))

sns.barplot(
    data=model_ranking,
    x="Macro F1",
    y="Model"
)

plt.xlim(0, 1)
plt.title("Baseline Model Comparison - Macro F1")
plt.xlabel("Macro F1 Score")
plt.ylabel("Model")

plt.show()

C:\Users\ABHINAV DASH\AppData\Local\Temp\ipykernel_21592\2113561062.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [41]:
# Plot accuracy comparison

accuracy_ranking = model_comparison.sort_values(
    by="Accuracy",
    ascending=False
)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=accuracy_ranking,
    x="Accuracy",
    y="Model"
)

plt.xlim(0, 1)
plt.title("Baseline Model Comparison - Accuracy")
plt.xlabel("Accuracy")
plt.ylabel("Model")

plt.show()

C:\Users\ABHINAV DASH\AppData\Local\Temp\ipykernel_21592\3236968492.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 16. Save Model Comparison Results

The baseline comparison table is saved as a CSV file.

This file can later be used during model selection and final project
documentation.

In [42]:
# Save model comparison results

comparison_file = "day_12_model_comparison.csv"

model_ranking.to_csv(
    comparison_file,
    index=False
)

print(f"Model comparison saved as: {comparison_file}")

Model comparison saved as: day_12_model_comparison.csv


In [43]:
# Verify saved file

saved_comparison = pd.read_csv(comparison_file)

print("Saved comparison file verified.")

display(saved_comparison)

Saved comparison file verified.


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Logistic Regression,0.669231,0.599889,0.507631,0.510613
1,KNN,0.638462,0.552284,0.472527,0.493093
2,Decision Tree,0.530769,0.446826,0.463919,0.450885
3,SVM,0.661538,0.469577,0.379365,0.346825
4,Random Forest,0.623077,0.437098,0.355678,0.327929


In [44]:
# Display final SVM summary

print("=" * 60)
print("DAY 12 - SVM MODEL SUMMARY")
print("=" * 60)

print(f"SVM Accuracy        : {svm_accuracy:.4f}")
print(f"SVM Macro Precision : {svm_precision:.4f}")
print(f"SVM Macro Recall    : {svm_recall:.4f}")
print(f"SVM Macro F1        : {svm_f1:.4f}")
print(f"SVM Train Accuracy  : {svm_train_accuracy:.4f}")
print(f"SVM Test Accuracy   : {svm_test_accuracy:.4f}")
print(f"Avg Confidence      : {average_confidence:.4f}")

DAY 12 - SVM MODEL SUMMARY
SVM Accuracy        : 0.6615
SVM Macro Precision : 0.4696
SVM Macro Recall    : 0.3794
SVM Macro F1        : 0.3468
SVM Train Accuracy  : 0.7303
SVM Test Accuracy   : 0.6615
Avg Confidence      : 0.6743


# Day 12 Findings

## Support Vector Machine Findings

1. A Support Vector Machine classifier was successfully developed for the
   student performance classification problem.

2. The model used an RBF kernel and was kept as an untuned baseline.

3. Categorical variables were transformed using One-Hot Encoding.

4. Numerical and ordinal features were standardized using StandardScaler.

5. The preprocessing and classification steps were combined into a single
   Scikit-learn Pipeline.

6. The dataset was divided using an 80/20 stratified train-test split.

7. The `G3` feature was removed before model training because the project
   target `performance_category` is directly derived from G3. This
   prevented target leakage.

8. SVM performance was evaluated using Accuracy, Macro Precision,
   Macro Recall and Macro F1-score.

9. The confusion matrix was used to examine correct and incorrect
   classifications for Low, Medium and High performance categories.

10. SVM was compared with Logistic Regression, Decision Tree, Random Forest
    and KNN using the same evaluation framework.

11. The best baseline model was identified using Macro F1-score.

12. No hyperparameter tuning was performed during Day 12. Parameters such
    as C, gamma and kernel configuration can be investigated during the
    later model tuning stage.

13. The baseline comparison results were saved in
    `day_12_model_comparison.csv`.

# Development Log - Day 12

## Date
07 October 2026

## Work Completed

- Loaded the UCI Student Performance dataset.
- Recreated the project-defined performance categories.
- Separated the target from the input features.
- Removed G3 to prevent target leakage.
- Defined numerical, ordinal, binary and nominal feature groups.
- Created an 80/20 stratified train-test split.
- Built an SVM-specific preprocessing pipeline.
- Applied StandardScaler to numerical and ordinal features.
- Applied One-Hot Encoding to categorical features.
- Trained an RBF-kernel SVM classifier.
- Evaluated the SVM using multiple classification metrics.
- Generated a classification report.
- Generated and visualized the confusion matrix.
- Compared training and testing accuracy.
- Generated prediction probabilities and confidence values.
- Compared SVM with four previous baseline models.
- Ranked all five models using Macro F1-score.
- Saved the model comparison results as a CSV file.

## Important Learning

SVM performance can be affected by the scale of input features.
Standardization was therefore introduced for numerical and ordinal
features.

The model was kept untuned so that future hyperparameter tuning can
provide a genuine performance improvement and comparison.

## Challenges

The main consideration during SVM development was ensuring that feature
scaling and categorical encoding were handled consistently without
allowing test-set information to influence training.

## Solution

A Scikit-learn ColumnTransformer and Pipeline were used to combine
preprocessing and model training into a reproducible workflow.

## Next Step

Continue baseline model analysis and prepare for cross-validation and
hyperparameter optimization in the later stages of the project.